# Softmax·교차엔트로피·CNN

신경망 출력의 확률 표현과 작은 영상의 합성곱·분류 과정 확인

코드 셀을 위에서 아래로 실행하고 문제의 조건을 바꾸어 결과 비교

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)

## 안정적인 Softmax

로짓 `[2,1,−1]`을 확률로 바꾸시오. 모든 로짓에 1,000을 더했을 때 확률이 같은지 확인하시오.

In [ ]:
def softmax(z):
    e=np.exp(z-np.max(z));return e/e.sum()
z=np.array([2.,1.,-1.]);prob=softmax(z)
print(prob)
assert np.allclose(softmax(z+1000),prob)
J=np.diag(prob)-np.outer(prob,prob)
eps=1e-6
numeric=np.column_stack([(softmax(z+eps*v)-softmax(z-eps*v))/(2*eps) for v in np.eye(3)])
assert np.allclose(J,numeric,atol=1e-8)
print('확률의 미분행렬:');print(J)

## 교차엔트로피의 기울기

정답 클래스가 0일 때 로짓에 대한 손실의 기울기를 구하시오. Softmax를 거치지 않은 로짓을 `cross_entropy`에 전달하시오.

In [ ]:
import torch
import torch.nn.functional as F
torch.set_num_threads(1)
zt=torch.tensor(z,dtype=torch.float64,requires_grad=True)
loss=F.cross_entropy(zt[None,:],torch.tensor([0]));loss.backward()
expected=prob-np.array([1.,0.,0.])
print('손실:',loss.item(),'기울기:',zt.grad.numpy())
assert np.allclose(zt.grad.numpy(),expected)

## 여러 출력: 붓꽃 세 종

꽃잎 너비·꽃받침 너비를 입력으로 받고 세 로짓을 출력하는 작은 신경망을 학습하시오. 전처리는 훈련 자료에만 맞추시오. 로짓·확률·예측 클래스가 각각 무엇인지 설명하시오.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix
iris=load_iris();features=iris.data[:,[3,1]]
Xtr,Xte,ytr,yte=train_test_split(features,iris.target,test_size=.25,stratify=iris.target,random_state=42)
scaler=StandardScaler().fit(Xtr)
train=torch.tensor(scaler.transform(Xtr),dtype=torch.float32)
test=torch.tensor(scaler.transform(Xte),dtype=torch.float32)
target=torch.tensor(ytr)
torch.manual_seed(42)
iris_model=torch.nn.Sequential(torch.nn.Linear(2,8),torch.nn.ReLU(),torch.nn.Linear(8,3))
optimizer=torch.optim.Adam(iris_model.parameters(),lr=.03)
for _ in range(300):
    optimizer.zero_grad();loss=F.cross_entropy(iris_model(train),target)
    loss.backward();optimizer.step()
with torch.no_grad():
    logits=iris_model(test);probabilities=logits.softmax(dim=1)
    predicted=logits.argmax(dim=1).numpy()
print('첫 시험 표본 로짓:',logits[0].numpy())
print('확률:',probabilities[0].numpy())
print('시험 정확도:',accuracy_score(yte,predicted))
print(confusion_matrix(yte,predicted))
assert torch.allclose(probabilities.sum(dim=1),torch.ones(len(test)))

## 작은 영상 두 장

6×6 O·X 영상을 입력→합성곱→ReLU→풀링→분류 순서로 처리하시오. 각 단계의 배열 크기를 실행 전에 예측하시오.

In [ ]:
o_image = [[0, 0, 1, 1, 0, 0],
           [0, 1, 0, 0, 1, 0],
           [1, 0, 0, 0, 0, 1],
           [1, 0, 0, 0, 0, 1],
           [0, 1, 0, 0, 1, 0],
           [0, 0, 1, 1, 0, 0]]
x_image = [[1, 0, 0, 0, 0, 1],
           [0, 1, 0, 0, 1, 0],
           [0, 0, 1, 1, 0, 0],
           [0, 0, 1, 1, 0, 0],
           [0, 1, 0, 0, 1, 0],
           [1, 0, 0, 0, 0, 1]]
shifted_x_image = [[0, 1, 0, 0, 0, 0],
                   [0, 0, 1, 0, 0, 1],
                   [0, 0, 0, 1, 1, 0],
                   [0, 0, 0, 1, 1, 0],
                   [0, 0, 1, 0, 0, 1],
                   [0, 1, 0, 0, 0, 0]]
shifted_o_image = [[0, 1, 1, 0, 0, 0],
                   [1, 0, 0, 1, 0, 0],
                   [0, 0, 0, 0, 1, 0],
                   [0, 0, 0, 0, 1, 0],
                   [1, 0, 0, 1, 0, 0],
                   [0, 1, 1, 0, 0, 0]]

In [ ]:
from torch import nn
torch.manual_seed(42)
X=torch.tensor([o_image,x_image],dtype=torch.float32)[:,None,:,:]
labels=torch.tensor([0,1])
model=nn.Sequential(nn.Conv2d(1,1,3),nn.ReLU(),nn.MaxPool2d(2),
                    nn.Flatten(),nn.Linear(4,1),nn.ReLU(),nn.Linear(1,2))
t=X
for layer in model:
    t=layer(t);print(type(layer).__name__,tuple(t.shape))
assert X.shape == (2,1,6,6)
# 첫 합성곱 출력 한 칸을 직접 계산
conv=model[0]
manual=(X[0,0,:3,:3]*conv.weight[0,0]).sum()+conv.bias[0]
assert torch.allclose(manual,conv(X)[0,0,0,0])
optimizer=torch.optim.Adam(model.parameters(),lr=.01)
losses=[]
for _ in range(700):
    optimizer.zero_grad();logits=model(X);loss=F.cross_entropy(logits,labels)
    loss.backward();optimizer.step();losses.append(loss.item())
print('처음·마지막 손실:',losses[0],losses[-1])
print('훈련 두 영상의 확률:',model(X).softmax(dim=1).detach().numpy())
assert losses[-1] < losses[0]
plt.plot(losses);plt.xlabel('Iteration');plt.ylabel('Cross entropy');plt.show()

## 위치가 달라진 영상

학습에 쓰지 않은 위치 변형 영상의 확률을 비교하시오. 두 장을 잘 맞혔다는 결과와 일반적인 인식 성능을 구분하시오.

In [ ]:
shifted=torch.tensor([shifted_o_image,shifted_x_image],dtype=torch.float32)[:,None,:,:]
with torch.no_grad(): print(model(shifted).softmax(dim=1).numpy())
fig,ax=plt.subplots(1,4,figsize=(8,2))
for a,im,title in zip(ax,[o_image,x_image,shifted_o_image,shifted_x_image],['O','X','Shifted O','Shifted X']):
    a.imshow(im,cmap='gray_r');a.set_title(title);a.axis('off')
plt.show()

O·X 수치 예의 출처: Josh Starmer, *The StatQuest Illustrated Guide to Neural Networks and AI* 및 [공개 예제](https://github.com/StatQuest/signa). 수치 예를 활용한 본 과목의 간결한 구현